# MIDI measurewise — per-measure lilylet / MIDI inspection

Reads a MIDI-measurewise dataset from a **config file** (`CondMidiPatchy` / `cond-midi.lmmw.pt`),
picks a **random sample**, picks a **random measure number** in it, then pulls out and decodes
the lilylet patches AND the midi event patches that belong to that measure — the lilylet side by
its own measure index `j`, the midi side by its lilylet-aligned `src_measure` (so a repeated
section's midi still lines up with the right lilylet measure).

Test config: `configs/midi-measurewise.yaml` (local `test20260629`).

In [1]:
from pathlib import Path
import sys, os

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'starry').is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import random
import torch

from starry.utils.config import Configuration
from starry.lilylet.data.patchifier import LilyletTokenizer
from starry.midi.tokenizer import MidiTokenizer
from starry.midi.data.condPatchy import _get_store

# --- choose the config + where its data.root lives (DATA_DIR base) ---
CONFIG = str(REPO_ROOT / 'configs' / 'midi-measurewise.yaml')
DATA_DIR = str(Path.home() / 'data')
SEED = None  # set an int to make the random sample/measure pick reproducible
print('config:', CONFIG)

config: /home/camus/work/deep-starry/configs/midi-measurewise.yaml


## 1. Resolve the dataset root from the config and open the store

In [2]:
config = Configuration.createOrLoad(CONFIG, volatile=True)
assert config['data.type'] == 'CondMidiPatchy', f"expected CondMidiPatchy, got {config['data.type']}"

ROOT = os.path.join(DATA_DIR, config['data.root'])
print('data.root :', config['data.root'])
print('resolved  :', ROOT)
print('exists    :', os.path.exists(ROOT))

store = _get_store(ROOT)
print('items     :', len(store))
print('tokenizers:', store.artifact['tokenizers'])

data.root : lilylet/midi-measurewise/test20260629/cond-midi.lmmw.pt
resolved  : /home/camus/data/lilylet/midi-measurewise/test20260629/cond-midi.lmmw.pt
exists    : True
items     : 215
tokenizers: {'lyl': {'vocab_size': 256}, 'midi': {'vocab_size': 41}}


## 2. Random sample + random measure

A sample's `measures` is each patch's OWN measure (lilylet `j` / midi `i`); `src_measures` is the
lilylet-aligned measure (lilylet: own `j`; midi: `source_measure(i)`). We pick a measure number
that exists on the lilylet side, then select lilylet patches by own-measure and midi patches by
`src_measure` — so under expanded repeats a played midi measure maps to its lilylet source measure.

In [3]:
rng = random.Random(SEED)

EX = rng.randrange(len(store))
item = store.get(EX)
mod = item['modality']
own = item['measures'].long()
src = item['src_measures'].long()
L = int(item['lyl_count'])
T = item['patches'].shape[0]

# lilylet body measures present in this sample (own measure > 0 on the lyl side)
lyl_measures = sorted(set(int(m) for m in own[:L].tolist() if m > 0))
MEAS = rng.choice(lyl_measures)

print(f'sample #{EX}: id={item["id"]}')
print(f'  T={T}  lyl_count L={L}  midi patches={T - L}  M_lyl={item["M_lyl"]}  M_midi={item["M_midi"]}')
print(f'  lilylet body measures: {lyl_measures[0]}..{lyl_measures[-1]} ({len(lyl_measures)} measures)')
print(f'  >>> picked measure number: {MEAS}')

sample #25: id=20250408_203843_Romantic_Liszt, Franz_Keyboard_postinst
  T=15702  lyl_count L=468  midi patches=15234  M_lyl=58  M_midi=58
  lilylet body measures: 1..58 (58 measures)
  >>> picked measure number: 35


## 3. Decode the lilylet patches of measure `MEAS`

In [4]:
lt = LilyletTokenizer(str(REPO_ROOT / 'assets' / 'lilylet-tokenizer.json'))
mt = MidiTokenizer()
patches = item['patches'].long()

def decode_lyl_patch(row):
    toks = [lt.text_by_id.get(int(t), '?') for t in row if int(t) not in (lt.pad_id, lt.bos_id, lt.eos_id)]
    return ''.join(toks)

# lilylet patches whose OWN measure == MEAS (lilylet segment: positions < L)
lyl_rows = [i for i in range(L) if int(own[i]) == MEAS]
print(f'lilylet patches for measure {MEAS}: {len(lyl_rows)} patches (rows {lyl_rows[0]}..{lyl_rows[-1]})\n')
lyl_text = ''.join(decode_lyl_patch(patches[i]) for i in lyl_rows)
print('--- decoded lilylet (concatenated patches) ---')
print(lyl_text)

lilylet patches for measure 35: 14 patches (rows 250..263)

--- decoded lilylet (concatenated patches) ---
[r:34/23]\staff "1" \key bf \major \time 6/8 \ppp r8 <df' df'>( <c c'> <bf bf'> <af af'> <gf gf'> \\
\staff "1" <af af'>2. \\
\staff "2" \clef "bass" r8 <gf af ef'>( <gf af ef'> <gf af ef'> <gf af ef'> <gf af ef'> \\
\staff "2" <c,, c'>2. |



## 4. Decode the MIDI event patches aligned to measure `MEAS`

Midi patches whose lilylet-aligned `src_measure == MEAS`. Under expanded repeats a lilylet
measure can map to MORE than one played midi measure (it was repeated), so we also show the
midi patches' own measure `i` to make the repeat visible.

In [5]:
# midi patches (positions >= L) whose src_measure == MEAS
midi_rows = [i for i in range(L, T) if int(src[i]) == MEAS]
midi_own = sorted(set(int(own[i]) for i in midi_rows))
print(f'midi patches with src_measure=={MEAS}: {len(midi_rows)} patches')
print(f'  -> played midi measure(s): {midi_own}' + ('  (repeated!)' if len(midi_own) > 1 else '') + '\n')

print(f'--- decoded MIDI events (src_measure=={MEAS}) ---')
for i in midi_rows:
    line = mt.decode_event(patches[i].tolist())
    tag = '<eom>' if int(patches[i, 0]) == mt.eom_id else ''
    print(f'  [row {i:5d}] midi_meas={int(own[i]):>3} {tag:5} {line!r}')

midi patches with src_measure==35: 88 patches
  -> played midi measure(s): [35]

--- decoded MIDI events (src_measure==35) ---
  [row  6994] midi_meas= 35       'control_change 1 0 2 10'
  [row  6995] midi_meas= 35       'note_on 0 0 38 10'
  [row  6996] midi_meas= 35       'control_change 0 0 2 10'
  [row  6997] midi_meas= 35       'note_on 0 0 44 10'
  [row  6998] midi_meas= 35       'control_change 0 0 2 10'
  [row  6999] midi_meas= 35       'note_on 0 0 24 10'
  [row  7000] midi_meas= 35       'control_change 0 0 2 10'
  [row  7001] midi_meas= 35       'note_on 0 0 30 10'
  [row  7002] midi_meas= 35       'control_change 0 0 2 10'
  [row  7003] midi_meas= 35       'control_change 1 0 40 0'
  [row  7004] midi_meas= 35       'control_change 1 0 40 7f'
  [row  7005] midi_meas= 35       'note_off ee 0 38'
  [row  7006] midi_meas= 35       'note_on 0 0 49 10'
  [row  7007] midi_meas= 35       'control_change 0 0 2 10'
  [row  7008] midi_meas= 35       'note_on 0 0 55 10'
  [row  7009] m

## 5. Side-by-side summary

In [6]:
print(f'sample #{EX}  ({item["id"]})')
print(f'measure {MEAS}:')
print(f'  lilylet : {len(lyl_rows)} patch(es)')
print(f'  midi    : {len(midi_rows)} patch(es) across played measure(s) {midi_own}')
print()
print('lilylet:')
print('  ' + lyl_text.replace(chr(10), chr(10) + '  '))
print('midi events:')
for i in midi_rows:
    print('  ' + mt.decode_event(patches[i].tolist()))

sample #25  (20250408_203843_Romantic_Liszt, Franz_Keyboard_postinst)
measure 35:
  lilylet : 14 patch(es)
  midi    : 88 patch(es) across played measure(s) [35]

lilylet:
  [r:34/23]\staff "1" \key bf \major \time 6/8 \ppp r8 <df' df'>( <c c'> <bf bf'> <af af'> <gf gf'> \\
  \staff "1" <af af'>2. \\
  \staff "2" \clef "bass" r8 <gf af ef'>( <gf af ef'> <gf af ef'> <gf af ef'> <gf af ef'> \\
  \staff "2" <c,, c'>2. |
  
midi events:
  control_change 1 0 2 10
  note_on 0 0 38 10
  control_change 0 0 2 10
  note_on 0 0 44 10
  control_change 0 0 2 10
  note_on 0 0 24 10
  control_change 0 0 2 10
  note_on 0 0 30 10
  control_change 0 0 2 10
  control_change 1 0 40 0
  control_change 1 0 40 7f
  note_off ee 0 38
  note_on 0 0 49 10
  control_change 0 0 2 10
  note_on 0 0 55 10
  control_change 0 0 2 10
  note_on 0 0 36 10
  control_change 0 0 2 10
  note_on 0 0 38 10
  control_change 0 0 2 10
  note_on 0 0 3f 10
  control_change 0 0 2 10
  note_off ef 0 49
  note_off 0 0 55
  note_off 0 0